# Water towers: snowpack-weighted runoff onset inside each tower

Per water-tower unit of Immerzeel et al. (2020), from the pixels *inside the tower polygon* (the mountain part of its
basin, not the whole basin): the snowpack-weighted mean onset anomaly per water year, its coverage, the
snowpack-weighted spring temperature anomaly, the typical year-to-year shift, the spring-temperature sensitivity
(OLS and Theil–Sen), and the cumulative share of the tower's snowpack whose runoff onset has occurred by each day of
the water year.

**Weights.** Each pixel is weighted by the ERA5-Land climatological peak snow water equivalent of its 0.1° cell (the
median over water years of the maximum monthly-mean SWE in the melt-season window), so the statistics describe the
snowpack rather than the area. **Pixels.** The public onset pyramid at level 6 (~5 km): annual onset and the 11-year
median.

```
reads                                                     writes
─────                                                     ──────
public pyramid level 6 (annual + median onset) ────┐       data/aggregation/<v>/pyramid_level6.nc            (cache)
ERA5-Land icechunk store (Azure) ──────────────────┤       data/aggregation/<v>/era5_peak_swe_spring_t.nc    (cache)
Immerzeel et al. 2020 tower polygons (Zenodo) ─────┴──►    results/<v>/water_tower_statistics.csv
                                                           results/<v>/water_tower_onset_curves.nc
```

The two caches need the Azure SAS token the first time (the pyramid is read through the production package).

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import scipy.stats as sps
import xarray as xr
from rasterio import features
from rasterio.transform import from_bounds

from gsro_analysis import era5, paths, results, settings

In [ ]:
config = settings.load_config()
VERSION = config.version
UNIT = 'river_basins'
aggregation_dir = paths.aggregation_dir(UNIT, VERSION)
results_dir = paths.resultsdir(UNIT, VERSION)

PYRAMID_LEVEL = 6
SPRING_MONTHS = ['spring_month_1', 'spring_month_2', 'spring_month_3']
MIN_TOWER_PIXELS = 200       # pyramid pixels with a median onset inside the tower
MIN_YEAR_PIXELS = 50         # pixels with a valid onset for a tower-year value
MIN_COVERAGE = 0.1           # snowpack-weighted share with a valid onset for a tower-year to count
MIN_YEARS = 8                # tower-years for a sensitivity fit
pyramid_path = aggregation_dir / f'pyramid_level{PYRAMID_LEVEL}.nc'
era5_cache_path = aggregation_dir / 'era5_peak_swe_spring_t.nc'

## 1. Inputs

In [ ]:
if not pyramid_path.exists():
    from global_snowmelt_runoff_onset.pyramid import open_pyramid_level
    pyramid_ds = open_pyramid_level(config, PYRAMID_LEVEL)[['runoff_onset', 'runoff_onset_median']].load()
    encoding = {v: {'zlib': True, 'complevel': 4, 'dtype': 'int16', '_FillValue': -1} for v in pyramid_ds.data_vars}
    pyramid_ds.drop_vars('spatial_ref', errors='ignore').to_netcdf(pyramid_path, encoding=encoding)
pyramid_ds = xr.open_dataset(pyramid_path)       # int16 day of water year, -1 = no onset
pyramid_ds

In [ ]:
if not era5_cache_path.exists():
    era5_ds = era5.open_era5_land(config)
    peak_swe = era5_ds['snow_depth_water_equivalent'].max('month').median('water_year').load()
    spring_t = era5_ds['temperature_2m'].sel(month=SPRING_MONTHS).mean('month').load()
    xr.Dataset({'swe_peak': peak_swe, 't_spring': spring_t}).to_netcdf(era5_cache_path)
era5_cache_ds = xr.open_dataset(era5_cache_path)  # climatological peak SWE (lat, lon); spring 2 m temperature (water_year, lat, lon)
era5_cache_ds

In [ ]:
towers_gdf = gpd.read_file(f"zip://{settings.water_towers_zip()}!{settings.WTU_TOWER_MEMBER}")
specs_df = settings.water_tower_specs()
towers_gdf

## 2. Pixels of each tower and their snowpack weights

In [ ]:
lat, lon = pyramid_ds['latitude'].values, pyramid_ds['longitude'].values
water_years = [int(y) for y in pyramid_ds['water_year'].values]
dlon, dlat = lon[1] - lon[0], lat[0] - lat[1]
transform = from_bounds(lon[0] - dlon / 2, lat[-1] - dlat / 2, lon[-1] + dlon / 2, lat[0] + dlat / 2, len(lon), len(lat))
tower_id_grid = features.rasterize(((g, int(i)) for g, i in zip(towers_gdf.geometry, towers_gdf['WTU_ID'])),
                                   out_shape=(len(lat), len(lon)), transform=transform, fill=0, dtype='int32')
elat, elon = era5_cache_ds['latitude'].values, era5_cache_ds['longitude'].values
row_in_era5 = np.clip(np.round((elat[0] - lat) / (elat[0] - elat[1])).astype(int), 0, len(elat) - 1)
col_in_era5 = np.clip(np.round((lon - elon[0]) / (elon[1] - elon[0])).astype(int), 0, len(elon) - 1)
peak_swe = era5_cache_ds['swe_peak'].values
spring_t = era5_cache_ds['t_spring'].values
median_onset = pyramid_ds['runoff_onset_median'].values.astype('float32')
annual_onset = pyramid_ds['runoff_onset'].values.astype('float32')
print(f'{(tower_id_grid > 0).sum()} pyramid pixels inside {towers_gdf.WTU_ID.nunique()} towers')

## 3. Per tower and water year

Anomaly = the year's onset minus the pixel's 11-year median, snowpack-weighted over the pixels with a valid onset
(at least `MIN_YEAR_PIXELS`); coverage = their share of the tower's snowpack weight; the spring temperature anomaly
is the snowpack-weighted mean over the same pixels' cells, minus its 11-year mean. A tower-year counts in the fits
with coverage ≥ `MIN_COVERAGE`; fits need `MIN_YEARS` years. The typical shift is the mean |anomaly|.

In [ ]:
DAYS = np.arange(1, 367)

def weighted_cdf(days, weights):
    order = np.argsort(days)
    cum = np.cumsum(weights[order]) / weights.sum()
    return np.interp(DAYS, days[order], cum, left=0, right=1)

rows, median_curves, year_curves = [], {}, {}
for tower in towers_gdf['WTU_ID'].astype(int):
    r, c = np.nonzero((tower_id_grid == tower) & np.isfinite(median_onset) & (median_onset >= 0))
    if len(r) < MIN_TOWER_PIXELS:
        continue
    w = peak_swe[row_in_era5[r], col_in_era5[c]]
    w = np.where(np.isfinite(w), w, 0.0)
    if w.sum() <= 0:
        continue
    m0 = median_onset[r, c]
    t = spring_t[:, row_in_era5[r], col_in_era5[c]]
    tw = np.where(np.isfinite(t), w[None, :], 0.0)
    t_anom = np.nansum(t * tw, axis=1) / tw.sum(axis=1)
    t_anom = t_anom - t_anom.mean()
    anomaly, coverage, curves = [], [], []
    for k in range(len(water_years)):
        o = annual_onset[k, r, c]
        valid = np.isfinite(o) & (o >= 0) & (w > 0)
        coverage.append(w[valid].sum() / w.sum())
        anomaly.append(np.average(o[valid] - m0[valid], weights=w[valid]) if valid.sum() >= MIN_YEAR_PIXELS else np.nan)
        curves.append(weighted_cdf(o[valid], w[valid]) if valid.sum() >= MIN_YEAR_PIXELS else np.full(len(DAYS), np.nan))
    anomaly, coverage = np.array(anomaly), np.array(coverage)
    ok = np.isfinite(anomaly) & (coverage >= MIN_COVERAGE)
    row = {'WTU_ID': tower, 'n_pixels': len(r), 'n_years': int(ok.sum()), 'snowpack_weighted_latitude': float(np.average(lat[r], weights=w)),
           'typical_shift_d': float(np.mean(np.abs(anomaly[ok]))) if ok.any() else np.nan}
    if ok.sum() >= MIN_YEARS:
        ols = sps.linregress(t_anom[ok], anomaly[ok])
        ts = sps.theilslopes(anomaly[ok], t_anom[ok])
        row.update(ols_slope=ols.slope, ols_p=ols.pvalue, pearson_r=ols.rvalue,
                   theil_sen_slope=ts[0], theil_sen_low=ts[2], theil_sen_high=ts[3])
    row.update({f'anomaly_WY{y}': a for y, a in zip(water_years, anomaly)})
    row.update({f'coverage_WY{y}': cv for y, cv in zip(water_years, coverage)})
    row.update({f'spring_temperature_anomaly_WY{y}': tt for y, tt in zip(water_years, t_anom)})
    rows.append(row)
    median_curves[tower] = weighted_cdf(m0[w > 0], w[w > 0])
    year_curves[tower] = np.array(curves)
towers_df = pd.DataFrame(rows).set_index('WTU_ID')
towers_df.insert(0, 'name', specs_df['WTU_name_short'].reindex(towers_df.index).values)
print(f'{len(towers_df)} towers, {int(towers_df.ols_slope.notna().sum())} with a sensitivity fit')
towers_df[['name', 'n_pixels', 'n_years', 'typical_shift_d', 'ols_slope', 'theil_sen_slope']].round(2)

In [ ]:
towers_df.reset_index().assign(**results.provenance(config)).to_csv(results_dir / 'water_tower_statistics.csv', index=False, float_format='%.4g')
ids = list(median_curves)
curves_ds = xr.Dataset({'median_map_share': (('water_tower', 'day_of_water_year'), np.array([median_curves[i] for i in ids])),
                        'year_share': (('water_tower', 'water_year', 'day_of_water_year'), np.array([year_curves[i] for i in ids]))},
                       coords={'water_tower': ids, 'water_year': water_years, 'day_of_water_year': DAYS})
curves_ds['median_map_share'].attrs['long_name'] = "snowpack-weighted share of the tower's pixels whose 11-year median onset has occurred"
curves_ds['year_share'].attrs['long_name'] = "snowpack-weighted share of the year's valid pixels whose onset has occurred"
curves_ds.attrs.update({'weights': 'ERA5-Land climatological peak SWE of the pixel cell', 'citation': settings.WTU_CITATION})
curves_ds.to_netcdf(results_dir / 'water_tower_onset_curves.nc')
curves_ds